In [1]:
import json
import pandas as pd

from datetime import datetime

def read_json_file(file_path):
    """Read a JSON file and return the data."""
    import json
    with open(file_path, 'r') as file:
        return json.load(file)
    
def has_availability_section(data):
    """Check if the data contains an 'availability' section."""
    return any('availability' in section['section'] for section in data.get('sections', []))

In [3]:
# read all the json files in the meta directory and combine them into a single dataframe
import os
def read_all_json_files(directory):
    """Read all JSON files in a directory and return a combined DataFrame."""
    all_data = []
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            file_path = os.path.join(directory, filename)
            data = read_json_file(file_path)
            if data:
                year = data.get('year')
                journal = data.get('journal')
                month = data.get('month')
                article_subtype = data.get('article_subtype')
                doi = data.get('doi')
                authors = data.get('authors', [])
                num_authors = len(authors)
                primary_institution = data.get('primary_institution')
                primary_region = data.get('primary_region')
                number_of_keywords = len(data.get('keywords', []))
                open_access = data.get('open_access')
                figure_number = data.get('figure_count')
                table_number = data.get('table_count')
                reference_count = data.get('reference_count')
                page_count = data.get('page_count')

                submission_date = data.get('submission_history', {}).get('received')
                acceptance_date = data.get('submission_history', {}).get('accepted')

                if submission_date is not None:
                    submission_date_dt = datetime.strptime(submission_date, '%Y-%m-%d')
                    acceptance_date_dt = datetime.strptime(acceptance_date, '%Y-%m-%d') if acceptance_date else None
                else:
                    submission_date_dt = None
                    acceptance_date_dt = None

                review_time_days = (acceptance_date_dt - submission_date_dt).days if submission_date_dt and acceptance_date_dt else None
                is_availability_section = has_availability_section(data)
                all_data.append({
                    'year': year,
                    'month': month,
                    'journal': journal,
                    'article_subtype': article_subtype,
                    'doi': doi,
                    'num_authors': num_authors,
                    'primary_institution': primary_institution,
                    'primary_region': primary_region,
                    'number_of_keywords': number_of_keywords,
                    'open_access': open_access,
                    'figure_number': figure_number,
                    'table_number': table_number,
                    'reference_count': reference_count,
                    'page_count': page_count,
                    'submission_date': submission_date_dt,
                    'acceptance_date': acceptance_date_dt,
                    'review_time_days': review_time_days,
                    'has_availability_section': is_availability_section
                })
    return pd.DataFrame(all_data)

df_all = read_all_json_files('meta')

df_all.to_csv("fla_csvs/meta.csv")